# Урок 25 — Алгоритми пошуку та хешування

**Модуль 3 · Python Advanced · Автор: Viktor Nikoriak**

---

## Про що цей урок?

Як розробник, ви маєте розглядати пошук та хешування не як алгоритми з підручника, а як **архітектурні стратегії управління продуктивністю**.

> **Головна ідея:** Пошук — це мистецтво знаходити дані. Хешування — це мистецтво **уникати пошуку взагалі**.

| Частина | Тема |
|---------|------|
| 1 | Два фундаментальні підходи: навігація vs телепортація |
| 2 | Лінійний пошук — $O(n)$: груба сила |
| 3 | Бінарний пошук — $O(\log n)$: поділяй і володарюй |
| 4 | Три архітектурні патерни пошуку в Python |
| 5 | Хешування — $O(1)$: математична телепортація |
| 6 | Внутрішня архітектура Python `dict`: 5 кроків |
| 7 | Колізії та їх вирішення: chaining vs open addressing |
| 8 | Hashable-об'єкти: `__hash__`, `__eq__` та незмінність |
| 9 | Пам'ять проти швидкості: компроміси хеш-таблиці |
| 10 | Питання для співбесіди |
| 11 | Підсумок |

---

---

## Частина 1: Два Фундаментальні Підходи

---

### Ментальна модель: «Усунення простору» проти «Математичної телепортації»

Як розробник, ви можете думати про пошук двома принципово різними способами.

**Стратегія А — Навігація по фізичному простору даних:**
Ви рухаєтесь по структурі даних, порівнюючи елементи, і поступово звужуєте простір пошуку. До цієї категорії відносяться лінійний та бінарний пошук.

**Стратегія Б — Математичне обчислення адреси:**
Ви не шукаєте взагалі. Ви використовуєте математику, щоб **одразу обчислити точну адресу**, де лежать ваші дані. Це хешування.

```
Навігація (Лінійний):  [1] [2] [3] [4] [5] [6] [7] [8] ...  ← перебір
Навігація (Бінарний):  [ середина? ] → відкидаємо половину → [ середина? ]
Телепортація (Hash):   key → hash("apple") → bucket #51 → знайдено!
```

| Стратегія | Складність | Вимога | Аналогія |
|-----------|------------|--------|----------|
| Лінійний пошук | $O(n)$ | Немає | Читати книгу сторінку за сторінкою |
| Бінарний пошук | $O(\log n)$ | Дані **відсортовані** | Пошук у паперовому словнику |
| Хешування | $O(1)$ | Ключ хешований | Помічник, що пам'ятає все напам'ять |

---

---

## Частина 2: Лінійний Пошук — $O(n)$

---

### Стратегія грубої сили

Лінійний пошук перевіряє кожен елемент послідовно від початку до кінця, поки не знайде збіг або не дійде до кінця колекції.

**Обчислювальна складність:** $O(n)$ — якщо список збільшується вдвічі, час пошуку у найгіршому випадку також збільшується вдвічі.

**Де використовується в Python:**
- Вбудований оператор `in` для **списків та кортежів** → прихований лінійний пошук
- Методи `list.index()` та `list.count()` → прихований лінійний пошук
- Функція `filter()` без попередньої індексації

**Коли доречний:**
- Дані повністю невідсортовані
- Дані дуже малі (< 100 елементів)
- Однократний пошук без необхідності повторень

---

In [ ]:
# ── Лінійний пошук: явна та неявна реалізації ──────────────────────────────

def linear_search(data: list, target) -> int:
    """
    Явна реалізація лінійного пошуку.
    Повертає індекс першого входження або -1 якщо не знайдено.
    Складність: O(n) — у найгіршому випадку перевіряємо всі n елементів.
    """
    for index, item in enumerate(data):  # перебір кожного елемента з індексом
        if item == target:
            return index  # знайдено — повертаємо позицію негайно
    return -1  # перебрали всі — елемент відсутній

# ── Демонстрація ─────────────────────────────────────────────────────────────
products = ["молоко", "яблука", "хліб", "апельсини", "кава"]

# Явний пошук через нашу функцію
idx = linear_search(products, "хліб")
print(f"linear_search() → 'хліб' знайдено на позиції: {idx}")

# Неявний O(n) через оператор 'in' для списку
print(f"'in' оператор → 'кава' in products: {'кава' in products}")

# Демонстрація O(n) деградації при зростанні даних
import timeit

for n in [1_000, 10_000, 100_000, 1_000_000]:
    # Шукаємо елемент, якого немає → найгірший випадок: O(n) перевірок
    setup = f'data = list(range({n}))'
    stmt = f'{n + 1} in data'  # шукаємо елемент поза списком
    t = timeit.timeit(stmt, setup=setup, number=100) / 100 * 1_000_000

    print(f"  n={n:>10,}  → {t:>8.1f} мкс")

print()
print("Висновок: час зростає пропорційно до n → це O(n)")


---

## Частина 3: Бінарний Пошук — $O(\log n)$

---

### Інтуїція: «Поділяй і Володарюй»

Уявіть, що ви шукаєте слово у паперовому словнику. Ви **ніколи** не починаєте з першої сторінки і не читаєте по одному слову. Ви відкриваєте посередині, бачите літеру і миттєво відкидаєте половину книги, де слова точно немає.

Бінарний пошук — це саме ця інтуїція, перекладена в алгоритм.

**Сувора вимога:** Дані мають бути **заздалегідь відсортовані**. Вся механіка відкидання половини покладається на гарантію порядку.

### Математика: Чому $O(\log n)$?

$O(\log n)$ — це кількість разів, яку можна поділити $n$ навпіл до отримання 1:

```
n = 512 елементів:
512 → 256 → 128 → 64 → 32 → 16 → 8 → 4 → 2 → 1
= 9 кроків = log₂(512) = 9 ✓
```

| Розмір даних | Кроки бінарного пошуку | Кроки лінійного (найгірший) |
|-------------|------------------------|----------------------------|
| 1 000 | ~10 | 1 000 |
| 1 000 000 | ~20 | 1 000 000 |
| 1 000 000 000 | ~30 | 1 000 000 000 |

### Два вказівники: `low` та `high`

Алгоритм підтримує **вікно пошуку** за допомогою двох вказівників:
- `low` — лівий край вікна (включно)
- `high` — правий край вікна (включно)
- `mid` — середина вікна, яку ми перевіряємо кожного кроку

```
[10, 20, 30, 40, 50, 60, 70]   target = 40
  ↑                       ↑
 low                     high

mid = (0+6)//2 = 3 → arr[3] = 40 → ЗНАЙДЕНО!
```

---

In [ ]:
# ── Бінарний пошук: ручна реалізація ────────────────────────────────────────

def binary_search(sorted_array: list, target) -> int:
    """
    Ітеративний бінарний пошук у відсортованому масиві.
    Складність: O(log n) — кожен крок відкидає половину простору пошуку.
    Повертає індекс або -1 якщо елемент не знайдено.
    """
    low = 0                        # лівий край вікна пошуку
    high = len(sorted_array) - 1  # правий край вікна пошуку

    while low <= high:             # поки вікно не звузилось до нуля
        mid = (low + high) // 2   # середина вікна (цілочисельне ділення)

        if sorted_array[mid] == target:
            return mid            # знайдено! повертаємо позицію

        elif target < sorted_array[mid]:
            # target у лівій половині → відкидаємо праву (high зсуваємо ліворуч)
            high = mid - 1

        else:
            # target у правій половині → відкидаємо ліву (low зсуваємо праворуч)
            low = mid + 1

    return -1  # вікно повністю звузилось, елемент не знайдено


# ── Трасування пошуку для наочності ─────────────────────────────────────────
def binary_search_traced(arr: list, target) -> int:
    """Те саме, але з детальним трасуванням кожного кроку."""
    low, high, step = 0, len(arr) - 1, 0

    while low <= high:
        mid = (low + high) // 2
        step += 1
        print(f"  Крок {step}: вікно=[{low}..{high}], mid={mid}, arr[mid]={arr[mid]}", end="")

        if arr[mid] == target:
            print(f"  ← ЗНАЙДЕНО!")
            return mid
        elif target < arr[mid]:
            print(f"  → відкидаємо праву половину")
            high = mid - 1
        else:
            print(f"  → відкидаємо ліву половину")
            low = mid + 1

    print(f"  → не знайдено за {step} кроків")
    return -1


# ── Демонстрація ─────────────────────────────────────────────────────────────
sorted_data = list(range(0, 200, 10))  # [0, 10, 20, ..., 190]
print(f"Масив (20 елементів): {sorted_data}")
print()

print("Пошук 140:")
binary_search_traced(sorted_data, 140)

print()
print("Пошук 135 (відсутній):")
binary_search_traced(sorted_data, 135)


In [ ]:
# ── Pythonic спосіб: модуль bisect ───────────────────────────────────────────
import bisect

# bisect — C-оптимізована реалізація бінарного пошуку зі стандартної бібліотеки
# bisect_left(arr, x) → індекс куди б вставити x для збереження порядку

sorted_prices = [10, 25, 30, 45, 60, 80, 95, 100]

def contains_bisect(sorted_arr: list, target) -> bool:
    """
    O(log n) пошук через bisect: Pythonic та надшвидкий.
    Знаходить позицію вставки → перевіряє чи там знаходиться target.
    """
    idx = bisect.bisect_left(sorted_arr, target)   # O(log n) пошук позиції
    return idx < len(sorted_arr) and sorted_arr[idx] == target  # перевірка збігу

print("=== bisect: пошук у відсортованому списку ===")
for price in [30, 45, 55, 100, 200]:
    result = contains_bisect(sorted_prices, price)
    print(f"  {price:>5} → {'знайдено ✓' if result else 'відсутній ✗'}")

print()

# bisect.insort: вставка з підтриманням сортування — O(n) через зсув, але O(log n) для пошуку позиції
leaderboard = [100, 250, 340, 500, 720]
new_score = 410
bisect.insort(leaderboard, new_score)
print(f"Лідерборд після вставки {new_score}: {leaderboard}")

print()

# Порівняння продуктивності: list 'in' vs bisect
import timeit

n = 1_000_000
sorted_list = list(range(n))

t_linear = timeit.timeit(
    stmt=f'{n + 1} in data',     # шукаємо відсутній елемент → найгірший випадок
    setup=f'data = list(range({n}))',
    number=100
) / 100 * 1_000_000

t_binary = timeit.timeit(
    stmt='bisect.bisect_left(data, target) == len(data)',
    setup=f'import bisect; data = list(range({n})); target = {n + 1}',
    number=100
) / 100 * 1_000_000

print(f"=== Порівняння для n={n:,} ===")
print(f"  'in' (лінійний) : {t_linear:>8.1f} мкс  → O(n)")
print(f"  bisect (бінарний): {t_binary:>8.2f} мкс  → O(log n)")
print(f"  Прискорення      : {t_linear / t_binary:.0f}x")


---

## Частина 4: Три Архітектурні Патерни Пошуку

---

Як розробник, вибір патерну пошуку — це стратегічне рішення про масштабованість.

### Патерн 1 — «Лінійне сканування» (Linear Scan)

**Суть:** Послідовна перевірка кожного елемента. Жодних вимог до структури даних.

**Де у Python:** `item in my_list`, `list.index()`, `list.count()`, `filter()`

**Коли застосовувати:** Невідсортовані дані, малі обсяги, однократний пошук.

### Патерн 2 — «Бінарний пошук» (Binary Search)

**Суть:** Поділяй і володарюй. Вимагає відсортованих даних, але дає $O(\log n)$.

**Де у Python:** `bisect` модуль — C-оптимізована реалізація.

**Коли застосовувати:** Частий пошук у фіксованій відсортованій послідовності. Пам'ять під контролем, список не змінюється.

### Патерн 3 — «Словниковий пошук» (Dictionary Lookup)

**Суть:** Математична телепортація. Ключ → хеш → пряма адреса в пам'яті. $O(1)$.

**Де у Python:** `dict`, `set`, `dict.get()`, `key in dict`

**Коли застосовувати:** Швидкість доступу — найвищий пріоритет. Можна пожертвувати пам'яттю.

```
Алгоритм вибору патерну:
───────────────────────────────────────────────────────────────────
              Чи відсортовані дані?
                    │
          ┌─────────┴─────────┐
         НІ                  ТАК
          │                   │
  Чи потрібна       Чи будуть часті
  висока швидкість?  повторні пошуки?
   │         │          │        │
  НІ        ТАК        НІ      ТАК
   │         │          │        │
Linear    Dict/Set   Linear   bisect
 O(n)      O(1)      O(n)    O(log n)
```

---

---

## Частина 5: Хешування — $O(1)$: Математична Телепортація

---

### Концепція: «Геніальна Помічниця»

Що якби замість того, щоб гортати книгу обліку товарів, у вас була помічниця, яка пам'ятає ціну кожного товару **напам'ять**? Ви називаєте товар — вона миттєво видає ціну, незалежно від того, скільки товарів у магазині.

Це і є хешування. Хеш-таблиця не **шукає** — вона **обчислює** точну адресу даних.

### Архітектура хеш-таблиці

Хеш-таблиця — це **розріджений масив** (масив із порожніми слотами, що називаються «кошиками» або buckets).

```
Хеш-функція: "apple" → hash() → 843920194
Модуль:      843920194 % 8 = 2

Масив buckets:
┌───┬───────────────────────┬───┬───┬───┬───┬───┐
│ 0 │         порожньо      │ 2 │ 3 │ 4 │ 5 │ 6 │
└───┴───────────────────────┴───┴───┴───┴───┴───┘
                              ↑
                        "apple" → 5  тут!
```

**Чому $O(1)$?** Оскільки обчислення адреси масиву займає однаковий час незалежно від розміру масиву. Знаємо адресу — миттєво переходимо туди. Жодного циклу, жодного сканування.

### Python `dict` та `set`: Хеш-таблиці під капотом

- **`dict`:** Кожен кошик зберігає: хеш-значення + вказівник на ключ + вказівник на значення
- **`set`:** Аналогічно до `dict`, але кошики зберігають **лише** вказівники на ключі (значень немає)

```python
# dict: O(1) вставка, O(1) пошук, O(1) видалення
my_dict = {"apple": 5, "banana": 3}   # два кошика заповнені

# set: O(1) перевірка наявності — fast membership test
my_set = {"alice", "bob", "carol"}   # O(1) vs O(n) для списку
```

---

In [ ]:
# ── Демонстрація функції hash() у Python ─────────────────────────────────────

print("=== hash() для різних типів даних ===")
print()

# Рядки (str)
for s in ["apple", "banana", "orange"]:
    h = hash(s)
    # Якщо розмір таблиці = 8, індекс = hash % 8
    bucket = h % 8
    print(f"  hash('{s}') = {h:>22}  →  bucket #{bucket}")

print()

# Числа (int) — цікавий факт: hash(n) == n для малих цілих
for n in [0, 1, 42, 100, -5]:
    print(f"  hash({n:>4}) = {hash(n)}")

print()

# Кортежі — хешуються (незмінні)
t = (1, 2, 3)
print(f"  hash((1,2,3)) = {hash(t)}")
print(f"  hash(('x','y')) = {hash(('x','y'))}")

print()

# Список — НЕ хешується (змінний!)
try:
    hash([1, 2, 3])
except TypeError as e:
    print(f"  hash([1,2,3]) → TypeError: {e}")

print()

# Демонстрація детермінованості: той самий ключ → той самий хеш
k = "python"
print(f"  hash('{k}') при кожному виклику: {hash(k)}, {hash(k)}, {hash(k)}")
print("  → детермінований: однаковий ключ = однаковий хеш (у межах процесу)")


In [ ]:
# ── Симуляція механізму пошуку у хеш-таблиці ────────────────────────────────

# Це спрощена ментальна модель того, що робить CPython під капотом dict

class SimpleDictSimulation:
    """
    Навчальна симуляція хеш-таблиці для розуміння принципу O(1).
    Реальний CPython dict значно складніший і оптимізованіший.
    """
    def __init__(self, capacity: int = 8):
        self._capacity = capacity           # кількість buckets (слотів)
        self._buckets = [None] * capacity   # розріджений масив
        self._size = 0

    def _get_bucket_index(self, key) -> int:
        """Обчислює індекс bucket через hash і modulo — ось де O(1) magic."""
        raw_hash = hash(key)                    # крок 1: hash(key)
        index = raw_hash % self._capacity       # крок 2: відображення на масив
        return index

    def set(self, key, value) -> None:
        """Вставка: одразу стрибаємо до потрібного bucket — O(1)."""
        idx = self._get_bucket_index(key)
        print(f"  SET '{key}': hash={hash(key)}, bucket=#{idx}")
        self._buckets[idx] = (key, value)   # спрощено: ігноруємо колізії
        self._size += 1

    def get(self, key):
        """Пошук: математично обчислюємо адресу — O(1). Жодного циклу!"""
        idx = self._get_bucket_index(key)
        bucket = self._buckets[idx]
        print(f"  GET '{key}': hash={hash(key)}, bucket=#{idx}", end="")
        if bucket is None:
            print(" → KeyError")
            return None
        if bucket[0] == key:
            print(f" → знайдено: {bucket[1]}")
            return bucket[1]
        print(" → KeyError (bucket зайнятий іншим ключем — потрібне probing)")
        return None


sim = SimpleDictSimulation(capacity=8)
print("=== Симуляція вставки ===")
sim.set("apple", 5)
sim.set("banana", 3)
sim.set("grape", 12)

print()
print("=== Симуляція пошуку ===")
sim.get("apple")
sim.get("grape")
sim.get("mango")   # відсутній

print()
print("Висновок: обидві операції — O(1), бо ми ОБЧИСЛЮЄМО адресу, не ШУКАЄМО")


# 🧠 Внутрішня архітектура Python `dict` (CPython) — поглиблений розбір

---

## 🔷 0. Загальна ідея

`dict` у Python — це:

> **хеш-таблиця з відкритою адресацією (open addressing) + probing + компактне зберігання**

Тобто:

* немає linked lists (як у Java старих версіях)
* все лежить у **масиві**
* колізії вирішуються через **пошук іншого слота (probe)**

---

## 🔷 1. Структура в пам’яті

У CPython (3.6+) dict має **розділену структуру**:

### 📦 Основні компоненти:

```text
dict
 ├── ma_keys   → таблиця ключів (hash + key)
 └── ma_values → масив значень (іноді NULL для shared dict)
```

---

### 🔹 Таблиця ключів (`ma_keys`)

Містить:

```text
index → entry
entry = (hash, key, value_index)
```

або (спрощено):

```python
class Entry:
    hash: int
    key: object
    value: object   # або індекс
```

---

### 🔹 Масив індексів (index table)

Це дуже важливо:

```text
[ -, -, 5, -, 2, -, ... ]
```

* тут лежать **індекси на entries**
* `-` означає empty
* спеціальні значення:

  * `EMPTY`
  * `DUMMY` (видалений елемент)

---

## 🔷 2. Крок 1 — Обчислення hash

```python
h = hash(key)
```

### ⚠️ Важливі деталі:

* для рядків → SipHash (захист від DoS)
* hash **стабільний в рамках процесу**
* але **random seed між запусками**

---

### 🧠 Інсайт

```text
hash ≠ index
```

hash → це просто велике число
index → це позиція в таблиці

---

## 🔷 3. Крок 2 — Обчислення індексу

```python
index = hash & (table_size - 1)
```

### Чому так?

Бо:

```text
table_size = 2^n
```

Тому:

```text
mod table_size == bitmask
```

👉 це **швидше ніж %**

---

### 🧠 Інсайт

```text
Python використовує тільки НИЖНІ біти hash
```

Тому важливо:

> hash-функція має добре перемішувати біти

---

## 🔷 4. Крок 3 — Перевірка bucket

```python
entry = table[index]
```

---

### Випадки:

#### ✅ EMPTY

```text
→ ключа немає
→ можна вставляти
```

---

#### ✅ MATCH

```python
if entry.hash == hash and entry.key == key:
```

👉 повертаємо значення

---

#### ❌ COLLISION

```text
інший ключ, але той самий bucket
```

👉 переходимо до probing

---

## 🔷 5. Крок 4 — Колізії (probing)

Python використовує:

> 🧠 **perturbation probing (не просто лінійний!)**

---

### 📌 Формула:

```python
perturb = hash

while True:
    index = (5 * index + 1 + perturb) & mask
    perturb >>= 5
```

---

### 🔥 Що тут відбувається?

#### 1. `5 * index + 1`

* базовий рух по таблиці
* не лінійний → менше clustering

---

#### 2. `perturb`

* додає randomness
* використовує інші біти hash

---

#### 3. `perturb >>= 5`

* поступово “згасає”
* алгоритм переходить до більш простого probing

---

### 🧠 Інсайт (важливо!)

```text
Python НЕ використовує:
- chaining
- простий linear probing
```

Він використовує:

> ⚡ hybrid probing → balance між cache locality і randomness

---

## 🔷 6. Видалення (DUMMY slots)

Коли ти робиш:

```python
del my_dict[key]
```

Python:

```text
НЕ очищає слот повністю
```

---

### Чому?

Бо інакше:

```text
probing chain зламається
```

---

### Тому:

```text
slot = DUMMY
```

---

### Наслідок:

* lookup продовжує працювати
* але таблиця “засмічується”

👉 тому потрібен resize

---

## 🔷 7. Resize (rehashing)

Python збільшує dict коли:

```text
load factor ≈ 2/3
```

---

### Що відбувається:

1. створюється нова таблиця (×2)
2. всі ключі **перехешуються**
3. DUMMY зникають

---

### 🧠 Інсайт

```text
resize = дорогий O(n)
але рідкісний → амортизовано O(1)
```

---

## 🔷 8. Чому dict швидкий

### 📌 Причини:

#### 1. O(1) доступ (в середньому)

```text
hash → index → value
```

---

#### 2. Cache locality

```text
дані лежать у масиві
```

👉 CPU кеш працює ефективно

---

#### 3. Немає pointer chasing

```text
(на відміну від linked list)
```

---

#### 4. Хороший probing

```text
мінімізує clustering
```

---

## 🔷 9. Чому інколи O(n)

### ❗ Worst case:

```text
всі ключі мають однаковий hash
```

👉 тоді:

```text
→ повний scan таблиці
```

---

### Але:

* SipHash → захист
* random seed → атаки складні

---

## 🔷 10. Реальна складність

| Операція | Середнє | Гірший випадок |
| -------- | ------- | -------------- |
| get      | O(1)    | O(n)           |
| set      | O(1)    | O(n)           |
| delete   | O(1)    | O(n)           |

---

## 🔷 11. Ментальна модель

Думай про dict так:

```text
dict = memory router
```

---

### Потік:

```text
key
 ↓
hash
 ↓
bitmask → index
 ↓
probe sequence
 ↓
slot
 ↓
value
```

---

## 🔷 12. Архітектурний висновок

```text
dict = баланс між:
- швидкістю (O(1))
- пам’яттю
- безпекою
```

---

## 🧠 Найважливіше (ядро)

```text
1. hash → НЕ індекс
2. index = hash & mask
3. collisions → probing (не chaining!)
4. DUMMY → для delete
5. resize → очищає систему
```

---

## 🚀 Якщо підняти ще на рівень вище

Ти вже бачиш:

> dict — це не “структура даних”

це:

```text
→ engine доступу до пам’яті
→ який перетворює ключ у позицію
```

---


---

## Частина 7: Колізії та Їх Вирішення

---

### Чому колізії неминучі?

Хеш-функція перетворює **нескінченну** множину можливих ключів (будь-які рядки) у **скінченну** множину bucket-індексів. За принципом «ящиків та кульок» (pigeonhole principle), рано чи пізно два різних ключі отримають однаковий індекс.

```
"Alice" → hash() → 843920194 % 8 = 2
"Bob"   → hash() → 113420194 % 8 = 2  ← Колізія!
```

### Стратегія 1: Метод ланцюжків (Chaining)

**Принцип:** Кожен bucket зберігає не одне значення, а **зв'язний список** значень. При колізії — просто додаємо до ланцюжка.

```
bucket #2: [("Alice", data_A)] → [("Bob", data_B)] → None
```

**Плюси:** Проста реалізація, коефіцієнт завантаження може перевищувати 100%.
**Мінуси:** Додаткова пам'ять на вказівники, втрата CPU cache locality.

**Використовується:** Java `HashMap`, Ruby `Hash`, старі реалізації Python (до 3.6 внутрішньо).

### Стратегія 2: Відкрита Адресація (Open Addressing)

**Принцип:** Всі елементи зберігаються **безпосередньо в масиві**. При колізії — знаходимо **наступний вільний slot** через зондування.

Три підвиди зондування:
| Вид | Послідовність кроків | Проблема |
|-----|---------------------|----------|
| **Лінійне** | +1, +2, +3 | «Кластеризація» (clustering) |
| **Квадратичне** | +1, +4, +9, +16 | Зменшена кластеризація |
| **Псевдовипадкове** | Перемішування бітів хешу | Рівномірний розподіл ✓ |

**Python використовує виключно псевдовипадкове зондування** — найефективніший підхід.

> **Типова помилка на співбесіді:** 90% кандидатів кажуть, що Python використовує chaining (ланцюжки). Насправді — open addressing з pseudo-random probing. Це демонструє глибоке знання мови.

---

In [ ]:
# ── Демонстрація колізій та деградації продуктивності ───────────────────────
import time

# Симуляція: що відбувається при збільшенні кількості колізій

class CollisionAwareHashTable:
    """Навчальна хеш-таблиця з підрахунком колізій."""

    def __init__(self, capacity: int):
        self.capacity = capacity
        self._table = [None] * capacity
        self.collisions = 0
        self.operations = 0

    def _probe(self, key, start_idx: int) -> int:
        """
        Лінійне зондування для наочності.
        Реальний Python використовує псевдовипадкове зондування.
        """
        idx = start_idx
        while self._table[idx] is not None and self._table[idx][0] != key:
            idx = (idx + 1) % self.capacity  # наступний slot (лінійно)
            self.collisions += 1
        return idx

    def insert(self, key, value) -> None:
        self.operations += 1
        start_idx = hash(key) % self.capacity
        final_idx = self._probe(key, start_idx)
        self._table[final_idx] = (key, value)

    def lookup(self, key):
        self.operations += 1
        start_idx = hash(key) % self.capacity
        idx = self._probe(key, start_idx)
        if self._table[idx] is not None:
            return self._table[idx][1]
        return None

    @property
    def load_factor(self) -> float:
        filled = sum(1 for s in self._table if s is not None)
        return filled / self.capacity


# ── Порівняємо: мало заповнена таблиця vs переповнена ───────────────────────
import random
random.seed(42)

words = [f"key_{i}" for i in range(100)]

print("Вплив коефіцієнта завантаження на кількість колізій:")
print(f"{'Розмір таблиці':>18} | {'Load Factor':>12} | {'Колізій':>10} | {'Колізій/оп':>12}")
print("-" * 62)

for capacity in [200, 150, 120, 110, 101]:
    ht = CollisionAwareHashTable(capacity)
    for w in words:
        ht.insert(w, True)
    print(f"{capacity:>18} | {ht.load_factor:>12.1%} | {ht.collisions:>10} | {ht.collisions/ht.operations:>12.2f}")

print()
print("Висновок: чим вища заповненість (load factor), тим більше колізій.")
print("Python тримає load factor < 66%, щоб уникнути деградації O(1) → O(n).")


---

## Частина 8: Hashable-Об'єкти: `__hash__`, `__eq__` та Незмінність

---

### Що значить «хешований» (hashable)?

Об'єкт є хешованим, якщо він виконує **три вимоги**:

1. Має метод `__hash__()`, що повертає ціле число, яке **ніколи не змінюється** протягом життя об'єкта
2. Має метод `__eq__()` для порівняння на рівність
3. Виконує **«Золоте правило хешування»:**

> Якщо `a == b` → то `hash(a) == hash(b)` **обов'язково**

### Контракт між `__hash__` та `__eq__`

Пошук у `dict` — це двоетапна перевірка:

```
Крок 1: hash(key)    → знаходимо правильний bucket (O(1))
Крок 2: key == found → перевіряємо чи це справді потрібний ключ (для колізій)
```

**Архітектурна пастка:** Якщо ви перевизначаєте `__eq__` у класі, Python **автоматично** встановлює `__hash__ = None`. Це означає, що ваш клас більше не можна використовувати як ключ у `dict` або елемент у `set`. Ви **зобов'язані** реалізувати `__hash__` самостійно.

### Чому незмінність (immutability) — критична вимога?

Уявіть що відбудеться, якщо ключ може змінитися після вставки:

```
my_list = [1, 2]
d[my_list] = "value"  # Python обчислює hash → кошик #42

my_list.append(3)     # вміст змінився → новий hash → кошик #99

d[my_list]  # Python шукає в кошику #99... але "value" в кошику #42!
            # Об'єкт назавжди "загублено" у власному словнику!
```

Саме тому Python **заборороняє** мутабельні об'єкти як ключі та викидає `TypeError`.

### Пастка кортежів

Кортеж хешованим є **лише тоді**, коли всі його елементи також хешовані:

```python
hash((1, 2, 3))    # ОК — всі елементи int
hash((1, [2, 3]))  # TypeError! — список всередині кортежу не хешований
```

---

In [ ]:
# ── Демонстрація hashable контракту ──────────────────────────────────────────

print("=== Вбудовані хешовані типи ===")
print(f"  hash('python')  = {hash('python')}")
print(f"  hash(42)        = {hash(42)}")
print(f"  hash(3.14)      = {hash(3.14)}")
print(f"  hash(True)      = {hash(True)}")
print(f"  hash((1,2,3))   = {hash((1,2,3))}")
print()

print("=== Нехешовані типи (мутабельні) ===")
for obj in [[1, 2], {1: 2}, {1, 2}]:
    try:
        hash(obj)
    except TypeError as e:
        print(f"  hash({obj!r}) → TypeError: {e}")
print()

print("=== Пастка кортежів ===")
print(f"  hash((1, 'a', True)) = {hash((1, 'a', True))}  ← всі immutable, OK")
try:
    hash((1, [2, 3]))
except TypeError as e:
    print(f"  hash((1, [2, 3]))  → TypeError: {e}  ← список всередині!")
print()

# Демонстрація Золотого правила хешування
print("=== Золоте правило: якщо a == b, то hash(a) == hash(b) ===")
# int та float: 1 == 1.0 → тому hash(1) == hash(1.0)
print(f"  1 == 1.0        → {1 == 1.0}")
print(f"  hash(1) == hash(1.0) → {hash(1) == hash(1.0)}  (відповідає правилу!)")
print(f"  hash(1) = {hash(1)}, hash(1.0) = {hash(1.0)}")
print()

# True == 1, тому hash(True) == hash(1)
print(f"  True == 1       → {True == 1}")
print(f"  hash(True) = {hash(True)}, hash(1) = {hash(1)}  ← однакові!")
print()
print("  Наслідок: True та 1 — це ОДИН і той самий ключ у словнику!")
d = {True: "boolean", 1: "integer"}
print(f"  dict(True='boolean', 1='integer') → {d}  ← залишився один ключ!")


In [ ]:
# ── Реалізація власного hashable класу ───────────────────────────────────────

class Vector2D:
    """
    2D-вектор, що може використовуватися як ключ у dict та елемент у set.
    Демонструє правильну реалізацію __hash__ та __eq__ контракту.
    """

    def __init__(self, x: float, y: float):
        # Приватні атрибути — запобігаємо зовнішній мутації
        self._x = x
        self._y = y

    @property
    def x(self) -> float: return self._x

    @property
    def y(self) -> float: return self._y

    def __eq__(self, other) -> bool:
        # Контракт рівності: порівнюємо тип та значення
        if not isinstance(other, Vector2D):
            return NotImplemented  # дозволяємо Python спробувати інший шлях
        return self._x == other._x and self._y == other._y

    def __hash__(self) -> int:
        # КРИТИЧНО: використовуємо ТІ САМІ поля що й __eq__
        # hash(tuple) — найкраща практика: делегуємо надійному алгоритму CPython
        return hash((self._x, self._y))

    def __repr__(self) -> str:
        return f"Vector2D({self._x}, {self._y})"


# ── Демонстрація: використання як ключ у dict ─────────────────────────────────
v1 = Vector2D(3.0, 4.0)
v2 = Vector2D(1.0, 0.0)
v3 = Vector2D(3.0, 4.0)  # дорівнює v1

print("=== Vector2D як ключ у словнику ===")
distance_cache = {
    v1: 5.0,   # |v1| = sqrt(9+16) = 5
    v2: 1.0,   # |v2| = 1
}

print(f"  distance_cache[v1] = {distance_cache[v1]}")
print(f"  distance_cache[v2] = {distance_cache[v2]}")
print()

# v1 == v3, тому hash(v1) == hash(v3) → той самий bucket
print(f"  v1 == v3 → {v1 == v3}")
print(f"  hash(v1) == hash(v3) → {hash(v1) == hash(v3)}")
print(f"  distance_cache[v3] = {distance_cache[v3]}  ← знайшов через v1!")
print()

# Використання як елемент множини
visited = {v1, v2, v3}  # v1 та v3 рівні → множина містить лише один
print(f"  visited = {{{v1}, {v2}, {v3}}}")
print(f"  len(visited) = {len(visited)}  ← v1 та v3 — один елемент")
print()

# ── Що відбувається якщо не реалізувати __hash__ при __eq__ ──────────────────
class BrokenVector:
    """Клас з __eq__ без __hash__ — Python заблокує хешування."""
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
    # __hash__ НЕ РЕАЛІЗОВАНО → Python встановить __hash__ = None

bv = BrokenVector(1, 2)
try:
    hash(bv)
except TypeError as e:
    print(f"  BrokenVector без __hash__: TypeError: {e}")
    print("  Python автоматично блокує хешування при перевизначенні __eq__!")


---

## Частина 9: Пам'ять проти Швидкості

---

### Фундаментальний компроміс хеш-таблиці

$O(1)$ продуктивність `dict` досягається за рахунок **навмисного марнування пам'яті**.

**Чому таблиця має бути розрідженою?**

Для того щоб псевдовипадкове зондування рідко зіштовхувалося з колізіями, хеш-таблиця **мусить мати достатньо порожніх слотів**. Python намагається підтримувати щонайменше **1/3 таблиці порожньою**.

Якщо словник заповнюється більш ніж на **2/3** (load factor > 0.67):
1. Python виділяє нову, більшу область пам'яті
2. **Перехешовує (rehash)** всі елементи — переобчислює індекси для нового розміру
3. Переносить всі дані у новий масив

**Наслідки Rehashing:**
- Після resize **всі ключі** змінюють свої фізичні позиції в масиві
- Саме тому **не можна змінювати `dict` під час ітерації** — `RuntimeError`
- Ітератор вказує на конкретні фізичні адреси, які після resize стають невалідними

### Порядок вставки у Python 3.7+

До Python 3.6 словники були **невпорядковані** (order залежав від внутрішнього хешу).
Починаючи з CPython 3.6 (і офіційно з 3.7), словники **зберігають порядок вставки**.

**Як це реалізовано без жертви O(1)?**
CPython 3.6+ використовує **компактне подвійне уявлення**:
- Окремий **масив індексів** (маленький, лише цілі числа)
- Окремий **компактний масив** (ключ, значення, хеш) у порядку вставки

Пошук іде через масив індексів (hash → індекс → позиція в компактному масиві) — O(1).
Ітерація йде по компактному масиву — в порядку вставки.

---

In [ ]:
# ── Аналіз пам'яті: list vs dict vs set ─────────────────────────────────────
import sys

print("=== Порівняння пам'яті: list vs set vs dict ===")
print()

for n in [10, 100, 1000]:
    my_list = list(range(n))
    my_set  = set(range(n))
    my_dict = {i: i for i in range(n)}

    size_list = sys.getsizeof(my_list)
    size_set  = sys.getsizeof(my_set)
    size_dict = sys.getsizeof(my_dict)

    print(f"n={n:>5}: list={size_list:>6} байт | set={size_set:>7} байт | dict={size_dict:>7} байт")
    print(f"         list=1x (базовий)  | set={size_set/size_list:.1f}x більше    | dict={size_dict/size_list:.1f}x більше")
    print()

print("Висновок: dict та set споживають значно більше пам'яті через розріджені bucket-масиви.")
print("Це свідомий компроміс: пам'ять → O(1) час.")

print()
print("=== Порядок вставки у Python 3.7+ ===")
d = {}
for key in ["banana", "apple", "cherry", "date", "elderberry"]:
    d[key] = len(key)

print(f"Вставлено: banana, apple, cherry, date, elderberry")
print(f"Ітерація : {list(d.keys())}")
print("Порядок вставки ЗБЕРЕЖЕНО — це гарантія Python 3.7+")


In [ ]:
# ── Кінцевий бенчмарк: лінійний vs бінарний vs хешовий пошук ────────────────
import timeit, bisect, sys

SIZES = [100, 1_000, 10_000, 100_000]

print(f"{'n':>8} | {'list O(n) мкс':>15} | {'bisect O(lgn)':>15} | {'dict O(1)':>12} | {'Прискорення dict/list':>22}")
print("-" * 80)

for n in SIZES:
    # Підготовка структур даних
    data_list   = list(range(n))
    data_dict   = {i: True for i in range(n)}
    target      = n - 1       # найгірший випадок для лінійного

    # Лінійний пошук
    t_list = timeit.timeit(
        stmt='target in data_list',
        globals={'data_list': data_list, 'target': target},
        number=10_000
    ) / 10_000 * 1_000_000

    # Бінарний пошук через bisect
    t_bisect = timeit.timeit(
        stmt='bisect.bisect_left(data_list, target) == target',
        globals={'bisect': bisect, 'data_list': data_list, 'target': target},
        number=10_000
    ) / 10_000 * 1_000_000

    # Хешовий пошук (dict)
    t_dict = timeit.timeit(
        stmt='target in data_dict',
        globals={'data_dict': data_dict, 'target': target},
        number=10_000
    ) / 10_000 * 1_000_000

    speedup = t_list / t_dict if t_dict > 0 else 0
    print(f"{n:>8,} | {t_list:>15.2f} | {t_bisect:>15.3f} | {t_dict:>12.3f} | {speedup:>22.1f}x")

print()
print("Висновок:")
print("  list O(n)  — лінійне зростання часу")
print("  bisect O(log n) — дуже повільне зростання")
print("  dict O(1)  — практично постійний час незалежно від n")


---

## Частина 10: Питання для Співбесіди

---

### Питання 1: Бінарний пошук на великому відсортованому списку

> **Запитання:** У вас є відсортований список з 10 мільйонів цілих чисел. Вам часто потрібно перевіряти, чи присутнє певне число. Як реалізувати це? Чому **не можна** просто використати оператор `in`?

**Коротка відповідь:**
Оператор `in` на списку — це $O(n)$ лінійне сканування, занадто повільне для великих даних. Оскільки список відсортований, потрібно використати бінарний пошук через модуль `bisect` — $O(\log n)$.

**Детальне пояснення:**
Лінійне сканування (`item in my_list`) перевіряє кожен елемент послідовно — у найгіршому випадку 10 мільйонів перевірок. Бінарний пошук ділить простір пошуку навпіл на кожному кроці. Для 10 мільйонів елементів — максимум **24 кроки**. Модуль `bisect` — C-оптимізована реалізація, готова до використання без написання циклів.

```python
import bisect

data = sorted_list_of_10_million_items
target = 10

# Правильний спосіб: O(log n)
index = bisect.bisect_left(data, target)
is_present = index < len(data) and data[index] == target
```

**Типова помилка:** Кандидати вважають, що `in` для списків магічно оптимізований до $O(1)$ або $O(\log n)$. Це не так — лише для `dict` та `set`.

---

### Питання 2: Чому `dict` — це $O(1)$?

> **Запитання:** Ми кажемо, що пошук у Python `dict` — це $O(1)$. Як Python досягає цього без перебору пам'яті?

**Коротка відповідь:**
Python взагалі не шукає. Він використовує математичну **хеш-функцію**, яка перетворює ключ на ціле число, і використовує це число як прямий індекс у масиві — телепортація до потрібної комірки.

**Детальне пояснення:**
Хеш-таблиця — масив із порожніми слотами. `my_dict["apple"]` → Python викликає `hash("apple")` → отримує число → бере найменш значущі біти → використовує як індекс масиву. Звернення до масиву за індексом займає однаковий час незалежно від розміру — ось звідки $O(1)$.

```python
# Концептуальна модель O(1) телепортації
key = "apple"
hash_value = hash(key)          # наприклад, 843920194
bucket_index = hash_value % 8   # прямий стрибок до індексу 2
# Python одразу йде в bucket_index — жодного циклу!
```

**Типова помилка:** Плутають Python `dict` з деревоподібними структурами (які $O(\log n)$). Не можуть пояснити, чому $O(1)$ — через пряму індексацію масиву за хешем.

---

### Питання 3: Вирішення хеш-колізій в CPython

> **Запитання:** Оскільки пам'ять скінченна, а ключів нескінченно багато, два різних ключі рано чи пізно попадуть в один bucket. Як CPython вирішує цю колізію?

**Коротка відповідь:**
CPython використовує **відкриту адресацію з псевдовипадковим зондуванням** — математично стрибає до різних, псевдовипадкових bucket'ів, поки не знайде вільний слот або потрібний ключ. Не ланцюжки (chaining)!

**Детальне пояснення:**
Більшість підручників вчать «chaining» — зв'язні списки всередині bucket'ів. CPython **не використовує chaining** — це втрата cache locality та додаткова пам'ять. Натомість елементи зберігаються прямо в масиві. При колізії Python бере інші біти хешу, застосовує C-макрос `perturb` для перемішування, отримує новий псевдовипадковий індекс і зондує далі.

```python
# Спрощена ментальна модель CPython probing
def locate_bucket(table, key):
    index = hash(key) % len(table)
    perturb = hash(key)
    while table[index] is not None:
        if table[index].key == key:
            return table[index].value
        # Колізія! Псевдовипадковий стрибок:
        perturb >>= 5
        index = (5 * index + perturb + 1) % len(table)
```

**Типова помилка:** 90% кандидатів відповідають «chaining», бо так вчать у загальних курсах CS. Знання про open addressing демонструє глибоке знання Python.

---

### Питання 4: Чому `list` не може бути ключем `dict`?

> **Запитання:** Чому Python кидає `TypeError` при спробі використати `list` як ключ `dict`, але дозволяє `tuple`? Яку архітектурну катастрофу запобігає Python?

**Коротка відповідь:**
Ключі `dict` мають бути **хешованими і незмінними**. Якщо мутувати список після вставки — його хеш зміниться, і елемент назавжди «загубиться» у неправильному bucket'і.

**Детальне пояснення:**
Золоте правило хешування: якщо `a == b`, то `hash(a) == hash(b)`. Якщо Python дозволив би `[1, 2]` як ключ, він обчислив би хеш і поклав у bucket #5. Після `my_list.append(3)` вміст змінився — новий хеш → bucket #12. Наступний пошук іде в bucket #12, але дані в bucket #5 — «сирота», назавжди недосяжний. Кортежі незмінні — їхній хеш гарантовано стабільний протягом всього життя.

```python
d = {}
d[(1, 2)] = "Valid"   # tuple — hashable ✓
try:
    d[[1, 2]] = "Crash"
except TypeError as e:
    print(e)  # unhashable type: 'list'
```

**Типова помилка:** «Списки занадто великі» або «кортежі швидші». Кандидати повністю пропускають архітектурну небезпеку мутації після вставки.

---

### Питання 5: Чому не можна змінювати `dict` під час ітерації?

> **Запитання:** Якщо додавати ключі до словника під час циклу `for`, Python кидає `RuntimeError`. Чому це заборонено на рівні пам'яті?

**Коротка відповідь:**
Додавання ключів може перевищити ліміт завантаженості (load factor), змусити Python виділити **більший масив** і перехешувати всі елементи — ітератор вказуватиме на знищену область пам'яті.

**Детальне пояснення:**
Для ефективного open addressing Python тримає ≥1/3 bucket'ів порожніми. При перевищенні 2/3 заповненості: виділяється новий більший масив → перераховуються всі індекси (`hash % new_size`) → всі елементи переміщуються. Ітератор вказує на конкретні фізичні адреси старого масиву — після resize вони вказують на «сміття». Saме тому `RuntimeError` — це захист на рівні пам'яті, не просто зручна перевірка.

```python
d = {'a': 1, 'b': 2, 'c': 3}

# RuntimeError: dictionary changed size during iteration
for key in d:
    if key == 'b':
        d['d'] = 4  # може викликати resize і перебудову всього масиву

# Правильний спосіб: ітеруємо по копії ключів
for key in list(d.keys()):
    if key == 'b':
        d['d'] = 4
```

**Типова помилка:** «Python захищає від нескінченних циклів». Насправді — захист від ітерації по знищеному масиву пам'яті.

---

---

## Підсумок Уроку

---

| Концепція | Складність | Вимоги | Типовий приклад у Python |
|-----------|-----------|--------|--------------------------|
| **Лінійний пошук** | $O(n)$ | Жодних | `x in my_list`, `list.index()` |
| **Бінарний пошук** | $O(\log n)$ | Відсортовані дані | `bisect.bisect_left()` |
| **Хешовий пошук** | $O(1)$ сер. | Hashable ключі | `x in my_dict`, `x in my_set` |

### Алгоритм вибору стратегії:

1. **Дані невідсортовані + однократний пошук** → `list.index()` / `in`
2. **Фіксовані відсортовані дані + часті пошуки** → `bisect` (менше пам'яті)
3. **Швидкість — пріоритет + ключі хешовані** → `dict`/`set` (більше пам'яті)

### Ключові архітектурні правила:

- `in` для `list` → $O(n)$, для `dict`/`set` → $O(1)$ — не плутати!
- Ключ `dict` = незмінний + `__hash__` + `__eq__` (відповідно до Золотого правила)
- При перевизначенні `__eq__` → обов'язково реалізувати `__hash__`
- Не змінювати `dict`/`set` під час ітерації → `RuntimeError`
- `dict` зберігає порядок вставки (Python 3.7+)

> **Головний Senior-інсайт:**
> Пошук — це боротьба з простором. Хешування — це уникнення цієї боротьби.
> Вибір між ними — це архітектурне рішення про компроміс між пам'яттю та часом.

---
*Наступний урок: Алгоритми сортування*
